# Connecting to OMERO at SURF



In [ ]:
import ezomero

# Import everything else up front, before opening the OMERO connection.
# omero-py's Ice communicator spawns background threads once connected, and
# importing heavier modules (matplotlib, ngff_zarr/itkwasm) *after* that can
# deadlock against Python's import lock. Doing all imports first avoids it.
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import ngff_zarr

In [ ]:
# Connect
host = 'omero1.fair-omero-lu.src.surf-hosted.nl'
port = 4064
user = 'nlbi'
secure = True
group = "training"
pw='analysis'

conn = ezomero.connect(host=host, port=port,user=user,group=group,secure=secure, password=pw)

In [ ]:
#List screens
screen_ids = ezomero.get_screen_ids(conn)    
for sid in screen_ids:
    screen_name = conn.getObject("Screen", sid).getName()
    print(f'Screen ID: {sid} | Name: {screen_name}')

In [ ]:
plate_ids = ezomero.get_plate_ids(conn, screen=1)
print(f'Plate IDs: {plate_ids} ')
well_ids = ezomero.get_well_ids(conn,plate=plate_ids[0])
print(f'Well IDs: {well_ids} ')

In [ ]:
plate = conn.getObject("Plate", plate_ids[0])

# listChildren returns wells unsorted, so sort by grid position
wells = sorted(plate.listChildren(), key=lambda w: (w.getRow(), w.getColumn()))
print(f"{len(wells)} wells")
for well in wells:
    print(well.getId(), well.getWellPos(), "fields:", len(list(well.listChildren())))

In [ ]:
#get dimensions of one image
well = wells[10]
image = well.getImage(0)          # first field of this well

print(f"{well.getWellPos()} — {image.getName()} (ID {image.getId()})")
print(f"X={image.getSizeX()} Y={image.getSizeY()} Z={image.getSizeZ()} "
      f"C={image.getSizeC()} T={image.getSizeT()} dtype={image.getPixelsType()}")

In [ ]:
z = image.getSizeZ() // 2
c, t = 0, 0

plane = image.getPrimaryPixels().getPlane(theZ=z, theC=c, theT=t)
print("plane shape:", plane.shape, plane.dtype, "range:", plane.min(), plane.max())

In [ ]:
z = image.getSizeZ() // 2
c, t = 1, 0

plane = image.getPrimaryPixels().getPlane(theZ=z, theC=c, theT=t)
print("plane shape:", plane.shape, plane.dtype, "range:", plane.min(), plane.max())

In [ ]:
#load one plane
z = image.getSizeZ() // 2
c, t = 1, 0

plane = image.getPrimaryPixels().getPlane(theZ=z, theC=c, theT=t)
print("plane shape:", plane.shape, plane.dtype, "range:", plane.min(), plane.max())

In [ ]:
import matplotlib.pyplot as plt

plane_norm = plane/plane.max()
plt.figure(figsize=(6, 6))
plt.imshow(plane_norm, cmap="gray")
plt.title(f"{well.getWellPos()} — z{z} c{c} t{t}")
plt.axis("off")
plt.show()

In [ ]:
import ngff_zarr

output_filename = 'output/output.ome.zarr'

multiscales = ngff_zarr.to_multiscales(plane_norm)
ngff_zarr.to_ngff_zarr(output_filename, multiscales)